# Example 14 — Lid-driven cavity, Re=100: the CFD validation classic

The most-cited benchmark in incompressible CFD (Ghia, Ghia & Shin, 1982): fluid in a unit
square, the top lid slides at $u=1$, all other walls no-slip. Steady Navier–Stokes:
$$\mathbf{u}\cdot\nabla\mathbf{u} = -\nabla p + \tfrac{1}{Re}\nabla^2\mathbf{u},\qquad \nabla\cdot\mathbf{u}=0,\qquad Re=100.$$
There is **no exact solution** — so this notebook builds its own reference (a 129²
streamfunction–vorticity finite-difference solver) and validates the PINN against it.

**PINN design:** outputs $(u,v,p)$; two momentum residuals + continuity; soft no-slip
walls with the lid $u=1$; and a **pressure anchor** $p(0,0)=0$ (with only velocity BCs,
pressure floats by a constant — Example 13 used a zero-mean gauge instead).

**Verified on an AMD Instinct MI210 (ROCm PyTorch):**
- FD reference (129², built in-notebook): $u_{min}$ on the vertical centerline = **−0.213** (Ghia: −0.206).
- PINN vs FD: centerline rel. L2 **u 0.027, v 0.031**; $u_{min}$ −0.193. FD 17 s, PINN 639 s (25k epochs).

The honest note: the singular top corners (lid meets stationary wall — a velocity
discontinuity) are where the residual stays large — the same strong-form difficulty as
shocks (Ex. 10) and Stokes' first problem (Ex. 16), and exactly where RBA (Ex. 7) helps.

In [ ]:
# Cell 1 -- Reference: streamfunction-vorticity finite differences (the Ghia setup)
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
RE = 100.0

def cavity_fd(n=129, max_iter=400000, tol=1e-8):
    h = 1.0/(n-1)
    psi = np.zeros((n, n)); om = np.zeros((n, n))
    dt = min(0.25*h*h*RE, 0.5*h)
    t0 = time.perf_counter()
    for it in range(max_iter):
        for _ in range(30):                       # psi Poisson: lap(psi) = -om
            psi[1:-1,1:-1] = 0.25*(psi[2:,1:-1]+psi[:-2,1:-1]+psi[1:-1,2:]+psi[1:-1,:-2] + h*h*om[1:-1,1:-1])
        u = np.zeros((n,n)); v = np.zeros((n,n))
        u[1:-1,1:-1] = (psi[1:-1,2:]-psi[1:-1,:-2])/(2*h)
        v[1:-1,1:-1] = -(psi[2:,1:-1]-psi[:-2,1:-1])/(2*h)
        u[:,-1] = 1.0                              # lid
        om_new = om.copy()                         # wall vorticity (Thom)
        om_new[1:-1,0]  = -2*psi[1:-1,1]/h**2
        om_new[1:-1,-1] = -2*psi[1:-1,-2]/h**2 - 2.0/h
        om_new[0,1:-1]  = -2*psi[1,1:-1]/h**2
        om_new[-1,1:-1] = -2*psi[-2,1:-1]/h**2
        adv = (u[1:-1,1:-1]*(om[2:,1:-1]-om[:-2,1:-1])/(2*h) + v[1:-1,1:-1]*(om[1:-1,2:]-om[1:-1,:-2])/(2*h))
        lap = (om[2:,1:-1]+om[:-2,1:-1]+om[1:-1,2:]+om[1:-1,:-2]-4*om[1:-1,1:-1])/h**2
        om_new[1:-1,1:-1] = om[1:-1,1:-1] + dt*(lap/RE - adv)
        d = np.max(np.abs(om_new-om))/(np.max(np.abs(om_new))+1e-12); om = om_new
        if d < tol and it > 1000:
            print(f'FD converged at it {it} ({time.perf_counter()-t0:.0f}s)'); break
    return psi, u, v

psi_fd, u_fd, v_fd = cavity_fd()
nfd = u_fd.shape[0]; g = np.linspace(0,1,nfd)
u_center_fd = u_fd[nfd//2,:]; v_center_fd = v_fd[:,nfd//2]
print(f'FD u_min (centerline) = {u_center_fd.min():.4f}   (Ghia Re=100: -0.2058)')

In [ ]:
# Cell 2 -- PINN: steady NS residuals + soft BCs + pressure anchor
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(2,96), nn.Tanh(), nn.Linear(96,96), nn.Tanh(),
                    nn.Linear(96,96), nn.Tanh(), nn.Linear(96,96), nn.Tanh(),
                    nn.Linear(96,3)).to(device)
def uvp(x, y):
    o = net(torch.cat([x, y], 1)); return o[:,0:1], o[:,1:2], o[:,2:3]
def grads(f, *xs):
    return [torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0] for x in xs]

EPOCHS = 25000
opt = torch.optim.Adam(net.parameters(), 1e-3)
zero2 = torch.zeros(1,1,device=device)
t0 = time.perf_counter()
for e in range(EPOCHS):
    if e == 15000:
        for gr in opt.param_groups: gr['lr'] = 3e-4
    if e == 21000:
        for gr in opt.param_groups: gr['lr'] = 8e-5
    opt.zero_grad()
    N = 4096
    x = torch.rand(N,1,device=device).requires_grad_(True)
    y = torch.rand(N,1,device=device).requires_grad_(True)
    u, v, p = uvp(x, y)
    ux, uy = grads(u, x, y); vx, vy = grads(v, x, y); px, py = grads(p, x, y)
    uxx = grads(ux, x)[0]; uyy = grads(uy, y)[0]; vxx = grads(vx, x)[0]; vyy = grads(vy, y)[0]
    rx = u*ux + v*uy + px - (uxx+uyy)/RE
    ry = u*vx + v*vy + py - (vxx+vyy)/RE
    rc = ux + vy
    s = torch.rand(512,1,device=device)
    wx = torch.cat([s, s, torch.zeros_like(s), torch.ones_like(s)])
    wy = torch.cat([torch.zeros_like(s), torch.ones_like(s), s, s])
    uw, vw, _ = uvp(wx, wy)
    lid = (wy > 0.999).float()
    bc = ((uw - lid)**2).mean() + (vw**2).mean()
    _, _, p0 = uvp(zero2, zero2)
    loss = (rx**2).mean() + (ry**2).mean() + 2*(rc**2).mean() + 10*bc + p0[0,0]**2
    loss.backward(); opt.step()
    if e % 5000 == 0: print(f'epoch {e:6d}  loss {loss.item():.2e}  ({time.perf_counter()-t0:.0f}s)')
if device.type == 'cuda': torch.cuda.synchronize()
print(f'\nPINN training: {time.perf_counter()-t0:.0f} s')

In [ ]:
# Cell 3 -- Validate against the FD reference
yt = torch.tensor(g, dtype=torch.float32, device=device).reshape(-1,1)
half = torch.full_like(yt, 0.5)
with torch.no_grad():
    u_c = uvp(half, yt)[0].cpu().numpy().ravel()
    v_c = uvp(yt, half)[1].cpu().numpy().ravel()
err_u = np.sqrt(np.mean((u_c-u_center_fd)**2)); err_v = np.sqrt(np.mean((v_c-v_center_fd)**2))
print(f'centerline rel L2:  u {err_u:.4f},  v {err_v:.4f}')

n = 101
xs = torch.linspace(0,1,n,device=device); X, Y = torch.meshgrid(xs, xs, indexing='ij')
with torch.no_grad(): U, V, _ = uvp(X.reshape(-1,1), Y.reshape(-1,1))
U = U.reshape(n,n).cpu().numpy(); V = V.reshape(n,n).cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(14.5,4.2))
xg = X.cpu().numpy(); ygr = Y.cpu().numpy()
c0 = ax[0].contourf(xg, ygr, np.sqrt(U**2+V**2), 21, cmap='viridis')
ax[0].streamplot(xs.cpu().numpy(), xs.cpu().numpy(), U.T, V.T, color='w', density=1.1, linewidth=0.6)
plt.colorbar(c0, ax=ax[0]); ax[0].set_title('PINN: speed + streamlines (Re=100)')
ax[0].set_xlabel('x'); ax[0].set_ylabel('y')
ax[1].plot(u_center_fd, g, 'g', lw=2.2, label='FD 129² (≈ Ghia)')
ax[1].plot(u_c, g, 'r--', lw=1.6, label=f'PINN (L2={err_u:.3f})')
ax[1].set_xlabel('u(0.5, y)'); ax[1].set_ylabel('y'); ax[1].legend(fontsize=9); ax[1].grid(alpha=.3)
ax[1].set_title('vertical centerline')
ax[2].plot(g, v_center_fd, 'g', lw=2.2, label='FD 129²')
ax[2].plot(g, v_c, 'r--', lw=1.6, label=f'PINN (L2={err_v:.3f})')
ax[2].set_xlabel('x'); ax[2].set_ylabel('v(x, 0.5)'); ax[2].legend(fontsize=9); ax[2].grid(alpha=.3)
ax[2].set_title('horizontal centerline')
plt.tight_layout(); plt.show()

## Observations (for fluid-dynamics notes)

- **The capstone flow:** everything from earlier examples appears here — steady NS (Ex. 12),
  a pressure field with a gauge (Ex. 13), soft BCs, and singular corners (Ex. 10/16).
- **It builds its own ground truth.** With no exact solution, the notebook ships a 129²
  streamfunction–vorticity solver whose centerline $u_{min}=-0.213$ sits right by Ghia's
  −0.206 — then scores the PINN against it. Always validate a PINN against *something*.
- **Pressure needs anchoring.** Only velocity BCs are given, so $p$ is defined up to a
  constant; the point anchor $p(0,0)=0$ fixes it (Ex. 13's zero-mean penalty is the
  alternative).
- **Where it's weakest is physical, not numerical:** the top corners are a genuine velocity
  discontinuity; the strong-form residual can't be zero there, so the PINN centerline is a
  touch shallow ($u_{min}$ −0.19 vs FD −0.21). This is precisely the regime RBA (Ex. 7) and
  domain decomposition (Ex. 9) were built for.
- **Verified on AMD MI210 (ROCm):** centerline rel-L2 ≈ 3% after 25k epochs (≈11 min).
  Higher Re (400, 1000) needs more capacity/epochs and benefits from curriculum training.

**Experiments to try:** raise Re to 400 (sharper corner vortices — watch the error grow);
add RBA weighting (Ex. 7) and check the corners; switch to a streamfunction output so
continuity is exact; recover Re from a few interior velocity samples (inverse, Ex. 3).